# Ontology-Constrained Memory (OCM) — Google Colab runner

A write-time **governed** memory layer for long-horizon LLM agents. This notebook runs the project end to end:

1. Get the code & install deps
2. Sanity tests
3. **Offline governance demo** (no GPU, no API key)
4. Benchmark + metrics (baselines B0–B3)
5. Full experiment suite (multi-seed CIs, significance, τ-sweep, stress)
6. *(optional)* Real embeddings (`sentence-transformers`)
7. *(optional)* Local **Qwen** extractor — in-process via `transformers`
8. *(optional)* Hosted OpenAI-compatible endpoint (no GPU)

Sections 1–5 run on a **CPU-only** runtime. Section 7 needs a **GPU** runtime (Runtime → Change runtime type → GPU; an A100 for the 32B model).

## 1. Get the code

**Option A — clone from GitHub** (public repo). **Option B** (next cell) — upload a zip or mount Google Drive.

In [ ]:
# Option A: clone from a Git remote.
import os, getpass
REPO_URL = "https://github.com/tysjosh/ocmr.git"  # public; for a private repo, supply a token below
REPO_DIR = "/content/ocmr"
TOKEN = getpass.getpass("GitHub token (press Enter for the public repo): ").strip()
url = REPO_URL.replace("https://", f"https://{TOKEN}@") if TOKEN else REPO_URL
if not os.path.exists(REPO_DIR):
    rc = os.system(f"git clone {url} {REPO_DIR}")
    if rc != 0:
        print("Clone failed — use Option B (upload/Drive) below.")
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    print("cwd:", os.getcwd())

In [ ]:
# Option B (only if you did NOT clone): upload a zip of the project, or mount Drive.
# from google.colab import files
# up = files.upload()                      # choose ocmr.zip
# !unzip -q ocmr.zip -d /content && ls /content
# %cd /content/ocmr                        # adjust if the zip nests a folder
#
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/ocmr          # adjust path

## 2. Install dependencies

The core (offline) demo needs only a few light packages. `chromadb` and `sentence-transformers` are **optional** — OCM falls back to a pure-Python vector index and a deterministic embedding provider when they are absent.

In [ ]:
# Light core install (enough for sections 2–5).
!pip -q install "pydantic>=2.6,<3" "networkx>=3.2" "fastapi>=0.110" "uvicorn>=0.29" "httpx>=0.27" "pytest>=8.0" "hypothesis>=6.100"

import sys, os
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import ocm
print("OCM importable from", ROOT)

## 3. Sanity tests

In [ ]:
!python -m pytest -q ocm/tests/test_has_status_assertions.py ocm/tests/test_experiment_stats.py ocm/tests/test_transformers_extractor.py

## 4. Offline governance demo (no GPU, no API key)

Facts are accepted, a status flip is **quarantined** (not silently overwritten), a correction **supersedes**, and the status query surfaces the contradiction inline.

In [ ]:
from ocm.core.config import Settings
from ocm.core.container import CoreContainer

c = CoreContainer(Settings(deterministic_test_mode=True, chroma_mode="memory", extractor="mock"))

def show(r, label):
    print(f"\n# {label}")
    print("  accepted   :", [o.candidate.predicate for o in r.accepted])
    print("  superseded :", [o.candidate.predicate for o in r.superseded])
    print("  quarantined:", [o.reason for o in r.quarantined])

show(c.write_pipeline.run("Alice owns Project Orion. Bob is assigned to Task T1.", "s1"), "W1 ownership + assignment")
show(c.write_pipeline.run("Bob completed Task T1.", "s2"), "W2 completion -> T1 done")
show(c.write_pipeline.run("Task T1 is not started.", "s3"), "W3 status flip -> QUARANTINED")
show(c.write_pipeline.run("Actually, Carol is assigned to Task T1.", "s4"), "W4 correction -> SUPERSEDE")

pkg = c.retrieval_pipeline.query("What is the current status of Task T1?", top_k=10)
print("\nQuery: 'What is the current status of Task T1?'")
print("  answer   :", pkg.answer)
print("  conflicts:", [{"accepted": cf.accepted, "quarantined": cf.quarantined, "reason": cf.reason} for cf in pkg.conflicts])

## 5a. Benchmark + metrics (baselines B0–B3)

In [ ]:
!python -m ocm.scripts.report_metrics --seed 1337

## 5b. Full experiment suite (multi-seed CIs, significance, τ-sweep, stress)

In [ ]:
!python -m ocm.scripts.run_experiments --quick

# Full protocol (slower):
# !python -m ocm.scripts.run_experiments --seeds 1337 7 42 99 2024 --per-category 6 --out results.json

## 6. (Optional) Real embeddings

Swaps the deterministic hashing embeddings for the real `all-MiniLM-L6-v2` model (downloads ~90 MB on first run). Runs on CPU or GPU.

In [ ]:
!pip -q install "sentence-transformers>=2.6"

from ocm.core.config import Settings
from ocm.core.container import CoreContainer

s = Settings(deterministic_test_mode=False, extractor="mock", embedding_mode="local",
             sqlite_path=":memory:", chroma_mode="memory")
c = CoreContainer(s)
c.write_pipeline.run("Alice owns Project Orion. Bob is assigned to Task T1.", "s1")
print("owner answer:", c.retrieval_pipeline.query("Who owns Project Orion?", top_k=5).answer)

## 7. (Optional) Local Qwen extractor — in-process via `transformers`

Loads Qwen **in the same process** (no server, no vLLM) and plugs it into OCM as the W1 extractor via `TransformersExtractor`.

**Model vs GPU:**
- **A100 80GB** → `Qwen/Qwen2.5-32B-Instruct` (bf16, ~64 GB) fits.
- **L4 / 40GB** → use 4-bit (`bitsandbytes`) or `Qwen/Qwen2.5-7B-Instruct`.
- **T4 16GB** → `Qwen/Qwen2.5-7B-Instruct` in 4-bit, or `Qwen/Qwen2.5-3B-Instruct`.

Set `MODEL_ID` to match your GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU — switch Runtime type to GPU."

In [ ]:
!pip -q install "transformers>=4.45" accelerate "bitsandbytes>=0.43"

In [ ]:
# Load Qwen in-process (mirrors a standard transformers load).
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-32B-Instruct"     # A100 80GB. Smaller GPU: "Qwen/Qwen2.5-7B-Instruct"
FOUR_BIT = False                            # set True on <40GB GPUs (uses bitsandbytes)

llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
kwargs = dict(torch_dtype=torch.bfloat16, device_map="auto")
if FOUR_BIT:
    from transformers import BitsAndBytesConfig
    kwargs.update(quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4"))
    kwargs.pop("torch_dtype", None)
llm_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)
print("Loaded", MODEL_ID)

In [ ]:
# Plug the local model into OCM as the W1 extractor and run the governed pipeline.
from ocm.extraction.transformers_extractor import TransformersExtractor
from ocm.core.container import CoreContainer
from ocm.core.config import Settings

extractor = TransformersExtractor(model=llm_model, tokenizer=llm_tokenizer, max_new_tokens=1024)
# deterministic embeddings + in-memory storage keep everything else hermetic; the LLM does W1.
c = CoreContainer(Settings(deterministic_test_mode=True, chroma_mode="memory"), extractor=extractor)

for t, label in [
    ("Alice owns Project Orion. Bob is assigned to Task T1.", "W1"),
    ("Bob completed Task T1.", "W2"),
    ("Task T1 is not started.", "W3"),
]:
    r = c.write_pipeline.run(t, label)
    print(label, "| accepted", [o.candidate.predicate for o in r.accepted],
          "| quarantined", [bool(o.reason) for o in r.quarantined])

pkg = c.retrieval_pipeline.query("Who owns Project Orion?", top_k=5)
print("owner:", pkg.answer)
pkg = c.retrieval_pipeline.query("What is the current status of Task T1?", top_k=10)
print("status:", pkg.answer, "| conflicts:", [(cf.accepted, cf.quarantined) for cf in pkg.conflicts])

## 8. (Optional) Hosted OpenAI-compatible endpoint (no GPU)

Reliable alternative to local serving, and the path the **experiment suite** uses for LLM runs (the suite builds many isolated containers, so it can't share one in-process model — a hosted endpoint serves them all). Works with OpenRouter / Together / Groq / your own server.

In [ ]:
import os
from ocm.core.config import Settings
from ocm.core.container import CoreContainer

BASE_URL = "https://openrouter.ai/api/v1"   # any OpenAI-compatible host
MODEL    = "qwen/qwen-2.5-7b-instruct"
API_KEY  = "sk-..."                          # <-- your key

s = Settings(extractor="llm", llm_base_url=BASE_URL, llm_model=MODEL, llm_api_key=API_KEY,
             llm_use_json_mode=True, deterministic_test_mode=True, chroma_mode="memory")
c = CoreContainer(s)
r = c.write_pipeline.run("Alice owns Project Orion. Bob is assigned to Task T1.", "s1")
print("accepted:", [o.candidate.predicate for o in r.accepted])
print("owner:", c.retrieval_pipeline.query("Who owns Project Orion?", top_k=5).answer)

# Full experiment suite over the hosted LLM (quick smoke):
# !OCM_LLM_API_KEY=$API_KEY python -m ocm.scripts.run_experiments --quick \
#   --extractor llm --llm-base-url $BASE_URL --llm-model "$MODEL" --embeddings deterministic

### Notes & caveats
- Sections 3–5 are fully offline/deterministic and need no GPU or keys.
- `Qwen2.5-32B-Instruct` (bf16 ~64 GB) needs an **A100 80GB**; on smaller GPUs set `FOUR_BIT=True` or use `Qwen2.5-7B-Instruct`.
- The in-process `TransformersExtractor` (section 7) is for an interactive demo. The multi-container **experiment suite** should use a hosted endpoint (section 8), since reloading a 32B model per container is impractical.
- Storage stays in-memory in every demo; nothing is written to disk except `benchmark.jsonl` / `results.json` when you ask for them.
- Free the GPU when done: `del llm_model; import gc, torch; gc.collect(); torch.cuda.empty_cache()`.